In [61]:
import numpy as np
import matplotlib.pyplot as plt
import taichi as ti
ti.init(arch=ti.cpu)
import tqdm

[Taichi] Starting on arch=x64


In [ ]:
a = 1.0
m = 1.0        


nx = 3
ny = 2
nz = 2
N_nodes = nx * ny * nz  # 12 

k = 100.0
k_diag = 100.0
ko = 80.0

eta = 0.02 * 2.0 * np.sqrt(k * m)
mu = 0.0

fmax = 0.1 * ko * a
amp = 1e-2

omega = np.sqrt(k / m)
periodicity = 2.0 * np.pi / omega
steps_per_period = 400
dt = periodicity / steps_per_period

total_step = 8000

print("N_nodes =", N_nodes)
print("dt =", dt)
print("periodicity =", periodicity)


N_nodes = 12
dt = 0.0015707963267948967
periodicity = 0.6283185307179586


In [ ]:

# ============================================================
# 3D Lattice 
# ============================================================
# Front (z=0) is nodes 0-5. Back (z=a) is nodes 6-11.
nodes_np = np.array([
    # Front 
    [0.0, 0.0, 0.0], [a, 0.0, 0.0], [2*a, 0.0, 0.0],  # 0, 1, 2
    [0.0, a,   0.0], [a, a,   0.0], [2*a, a,   0.0],  # 3, 4, 5
    # Back 
    [0.0, 0.0, a],   [a, 0.0, a],   [2*a, 0.0, a],    # 6, 7, 8
    [0.0, a,   a],   [a, a,   a],   [2*a, a,   a],    # 9, 10, 11
], dtype=np.float32)

##

vel_np = np.zeros((N_nodes, 3), dtype=np.float32)

# frames
bonds_np = np.array([
    # Front 
    [0,1], [1,2], [3,4], [4,5], [0,3], [1,4], [2,5],
    # Back 
    [6,7], [7,8], [9,10], [10,11], [6,9], [7,10], [8,11],
    # Z-axis
    [0,6], [1,7], [2,8], [3,9], [4,10], [5,11]
], dtype=np.int32)

# zodat het niet in elkaar valt
diag_bonds_np = np.array([
    # Front
    [0,4], [1,3], [1,5], [2,4],
    # Back
    [6,10], [7,9], [7,11], [8,10],
    # Top
    [3,10], [4,9], [4,11], [5,10],
    # Bottom
    [0,7], [1,6], [1,8], [2,7],
    # Sides
    [0,9], [3,6], [2,11], [5,8],
    #inner
    [1,10], [4,7]
], dtype=np.int32)


active_bonds_np = np.array([
    [0, 4],
    [1, 5],
], dtype=np.int32)

N_bonds = len(bonds_np)
N_diag_bonds = len(diag_bonds_np)
N_active_bonds = len(active_bonds_np)

# 3D rest lengths
rest_length_np = np.array([
    np.linalg.norm(nodes_np[j] - nodes_np[i])
    for i, j in bonds_np
], dtype=np.float32)

diag_rest_length_np = np.array([
    np.linalg.norm(nodes_np[j] - nodes_np[i])
    for i, j in diag_bonds_np
], dtype=np.float32)

active_rest_length_np = np.array([
    np.linalg.norm(nodes_np[j] - nodes_np[i])
    for i, j in active_bonds_np
], dtype=np.float32)

print("N_bonds =", N_bonds)
print("N_diag_bonds =", N_diag_bonds)
print("N_active_bonds =", N_active_bonds)

N_bonds = 20
N_diag_bonds = 22
N_active_bonds = 2


In [ ]:
# Preturbation
nodes_np[4, 1] += amp
nodes_np[4, 2] += amp * 0.5 

###############################
pos = ti.Vector.field(3, dtype=ti.f32, shape=N_nodes)
vel = ti.Vector.field(3, dtype=ti.f32, shape=N_nodes)
force = ti.Vector.field(3, dtype=ti.f32, shape=N_nodes)

bonds = ti.Vector.field(2, dtype=ti.i32, shape=N_bonds)
diag_bonds = ti.Vector.field(2, dtype=ti.i32, shape=N_diag_bonds)
active_bonds = ti.Vector.field(2, dtype=ti.i32, shape=N_active_bonds)

rest_length = ti.field(dtype=ti.f32, shape=N_bonds)
diag_rest_length = ti.field(dtype=ti.f32, shape=N_diag_bonds)
active_rest_length = ti.field(dtype=ti.f32, shape=N_active_bonds)

active_extension = ti.field(dtype=ti.f32, shape=N_active_bonds)
active_tension = ti.field(dtype=ti.f32, shape=N_active_bonds)

pos_record = ti.Vector.field(3, dtype=ti.f32, shape=(total_step, N_nodes))
vel_record = ti.Vector.field(3, dtype=ti.f32, shape=(total_step, N_nodes))
active_extension_record = ti.field(dtype=ti.f32, shape=(total_step, N_active_bonds))
active_tension_record = ti.field(dtype=ti.f32, shape=(total_step, N_active_bonds))

pos.from_numpy(nodes_np)
vel.from_numpy(vel_np)
bonds.from_numpy(bonds_np)
diag_bonds.from_numpy(diag_bonds_np)
active_bonds.from_numpy(active_bonds_np)
rest_length.from_numpy(rest_length_np)
diag_rest_length.from_numpy(diag_rest_length_np)
active_rest_length.from_numpy(active_rest_length_np)

@ti.func
def clamp(x, xmin, xmax):
    return ti.min(ti.max(x, xmin), xmax)

@ti.kernel
def zero_force():
    for i in range(N_nodes):
        
        force[i] = ti.Vector([0.0, 0.0, 0.0])

@ti.kernel
def nodal_f_passive():
    for b in range(N_bonds):
        i = bonds[b][0]
        j = bonds[b][1]
        rij = pos[j] - pos[i]
        lij = rij.norm()
        if lij > 1e-8:
            eij = rij / lij
            dl = lij - rest_length[b]
            vij = vel[j] - vel[i]
            dl_dt = vij.dot(eij)
            T = k * dl + eta * dl_dt
            force[i] += T * eij
            force[j] -= T * eij

@ti.kernel
def nodal_f_diag():
    for b in range(N_diag_bonds):
        i = diag_bonds[b][0]
        j = diag_bonds[b][1]
        rij = pos[j] - pos[i]
        lij = rij.norm()
        if lij > 1e-8:
            eij = rij / lij
            dl = lij - diag_rest_length[b]
            vij = vel[j] - vel[i]
            dl_dt = vij.dot(eij)
            T = k_diag * dl + eta * dl_dt
            force[i] += T * eij
            force[j] -= T * eij

@ti.kernel
def compute_active_extension():
    for b in range(N_active_bonds):
        i = active_bonds[b][0]
        j = active_bonds[b][1]
        rij = pos[j] - pos[i]
        lij = rij.norm()
        active_extension[b] = lij - active_rest_length[b]

@ti.kernel
def nodal_f_active():
   
    d1 = active_extension[0]
    d2 = active_extension[1]

    T1 = ko * d2
    T2 = -ko * d1

    T1 = clamp(T1, -fmax, fmax)
    T2 = clamp(T2, -fmax, fmax)

    active_tension[0] = T1
    active_tension[1] = T2

    i0, j0 = active_bonds[0][0], active_bonds[0][1]
    r0 = pos[j0] - pos[i0]
    l0 = r0.norm()
    if l0 > 1e-8:
        e0 = r0 / l0
        force[i0] += T1 * e0
        force[j0] -= T1 * e0

    i1, j1 = active_bonds[1][0], active_bonds[1][1]
    r1 = pos[j1] - pos[i1]
    l1 = r1.norm()
    if l1 > 1e-8:
        e1 = r1 / l1
        force[i1] += T2 * e1
        force[j1] -= T2 * e1

@ti.kernel
def update():
    for i in range(N_nodes):
        
        force[i] += -mu * vel[i]
        vel[i] += dt * force[i] / m
        pos[i] += dt * vel[i]

@ti.kernel
def data_record(frame: ti.i32):
    for i in range(N_nodes):
        pos_record[frame, i] = pos[i]
        vel_record[frame, i] = vel[i]
    for b in range(N_active_bonds):
        active_extension_record[frame, b] = active_extension[b]
        active_tension_record[frame, b] = active_tension[b]

In [ ]:

# ============================================================
# Simulation Loop
# ============================================================
pos.from_numpy(nodes_np)
vel.from_numpy(vel_np)

zero_force()
nodal_f_passive()
nodal_f_diag()
compute_active_extension()
nodal_f_active()
data_record(0)

for ii in tqdm.tqdm(range(total_step - 1), desc="Simulating"):
    zero_force()
    nodal_f_passive()
    nodal_f_diag()
    compute_active_extension()
    nodal_f_active()
    update()
    
    compute_active_extension()
    data_record(ii + 1)

Simulating: 100%|██████████| 7999/7999 [00:08<00:00, 904.55it/s]


In [ ]:

# ============================================================

bonds_index = ti.field(dtype=ti.i32, shape=N_bonds * 2)
diag_bonds_index = ti.field(dtype=ti.i32, shape=N_diag_bonds * 2)
active_bonds_index = ti.field(dtype=ti.i32, shape=N_active_bonds * 2)

bonds_index.from_numpy(bonds_np.flatten().astype(np.int32))
diag_bonds_index.from_numpy(diag_bonds_np.flatten().astype(np.int32))
active_bonds_index.from_numpy(active_bonds_np.flatten().astype(np.int32))

##############
pos_render = ti.Vector.field(3, dtype=ti.f32, shape=N_nodes)

def video(plt_step=1, close=0):
    pos_np = pos_record.to_numpy()[::plt_step]
    N_step = len(pos_np)

    window = ti.ui.Window(
        f"3D Two-Cube Odd Model: ko={ko}",
        (800, 600),
        vsync=True,
        fps_limit=60,
    )
    canvas = window.get_canvas()
    canvas.set_background_color((1, 1, 1))

    scene = window.get_scene()
    camera = ti.ui.Camera()
    gui = window.get_gui()

    pause = False
    step = 0

    while window.running:
        # Dynamic 3D Camera tracking the center of mass
        center_x = np.mean(pos_np[step, :, 0])
        center_y = np.mean(pos_np[step, :, 1])
        center_z = np.mean(pos_np[step, :, 2])

        camera.position(center_x + 1.5*a, center_y + 1.5*a, center_z + 4.0*a)
        camera.lookat(center_x, center_y, center_z)
        camera.up(0.0, 1.0, 0.0)
        scene.set_camera(camera)

        scene.ambient_light((0.7, 0.7, 0.7))
        scene.point_light(pos=(center_x + 2*a, center_y + 2*a, center_z + 3*a), color=(1, 1, 1))

        for e in window.get_events(ti.ui.PRESS):
            if e.key == ti.ui.ESCAPE: window.running = False
            elif e.key == ti.ui.SPACE: pause = not pause

        if window.is_pressed("r"): step = 0
        if window.is_pressed("z"): step += 10
        if window.is_pressed("x"): step -= 10
        step = max(0, min(step, N_step - 1))

        with gui.sub_window("Simulation", 0.00, 0.88, 0.45, 0.12) as w:
            w.text(f"frame: {step}/{N_step - 1}")
            w.text(f"physical step: {step * plt_step}")

        
        pos_render.from_numpy(pos_np[step])

        
        scene.lines(vertices=pos_render, indices=diag_bonds_index, color=(0.8, 0.8, 0.8), width=1.5)
        
        
        scene.lines(vertices=pos_render, indices=bonds_index, color=(0.1, 0.1, 0.1), width=4.0)

        #
        scene.lines(vertices=pos_render, indices=active_bonds_index, color=(0.9, 0.45, 0.0), width=6.0)

        # Render nodes
        scene.particles(pos_render, radius=0.06 * a, color=(0.1, 0.2, 0.9))

        canvas.scene(scene)
        window.show()

        if not pause: step += 1
        if step >= N_step:
            if close == 1: window.running = False
            else:
                step = N_step - 1
                pause = True

video(plt_step=10, close=0)